# Abalone Age Regression

**Author: Feras Elghandour**

Estimate abalone ring counts from physical measurements and compare model complexity against multiple error measures.

Train-fitted numeric/categorical preprocessing, a linear baseline, five-fold random-forest grid search, residual plots, feature importance, and sex-group error analysis.

Read README.md for result scope and DATA.md for input preparation. Run cells in order in a clean environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

print(f"numpy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")
print(f"matplotlib:   {plt.matplotlib.__version__}")
print(f"seaborn:      {sns.__version__}")

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 6)})

In [ ]:
df = pd.read_csv("abalone.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
df.info()
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nSex distribution:\n{df['Sex'].value_counts()}")

In [ ]:
# Descriptive statistics
df.describe().round(4)

In [ ]:
# Check for anomalous values
print(f"Height = 0 count: {(df['Height'] == 0).sum()}")
print(f"Height outliers (>0.5): {(df['Height'] > 0.5).sum()}")

# Remove height = 0 (physically impossible, measurement error)
df = df[df["Height"] > 0].copy()
print(f"\nAfter removing height=0: {df.shape[0]} rows")

In [ ]:
# Define features and target
target = "Rings"
num_features = ["Length", "Diameter", "Height", "Whole_weight",
                "Shucked_weight", "Viscera_weight", "Shell_weight"]
cat_features = ["Sex"]

X = df[num_features + cat_features]
y = df[target]

print(f"Features: {num_features + cat_features}")
print(f"Target: {target}")
print(f"\nTarget distribution:")
print(y.describe().round(2))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTarget (train) - mean: {y_train.mean():.2f}, std: {y_train.std():.2f}")
print(f"Target (test)  - mean: {y_test.mean():.2f}, std: {y_test.std():.2f}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_features),
    ]
)
print("Preprocessor configured:")
print(f"  Numeric: StandardScaler on {len(num_features)} features")
print(f"  Categorical: OneHotEncoder on {cat_features} (drop first → 2 dummies)")

In [ ]:
baseline = Pipeline([("pre", preprocessor), ("model", LinearRegression())])
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

bl_metrics = {
    "MAE": mean_absolute_error(y_test, y_pred_bl),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_bl)),
    "R²": r2_score(y_test, y_pred_bl),
}
print("=== Baseline: Linear Regression ===")
for k, v in bl_metrics.items():
    print(f"  {k}: {v:.4f}")

cv_r2 = cross_val_score(baseline, X_train, y_train, cv=5, scoring="r2")
print(f"  CV R² (5-fold): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

In [ ]:
rf_pipe = Pipeline([("pre", preprocessor), ("model", RandomForestRegressor(random_state=42))])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
}

grid = GridSearchCV(rf_pipe, param_grid, cv=5, scoring="r2", n_jobs=2, verbose=0)
grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV R²:  {grid.best_score_:.4f}")

In [ ]:
best_rf = grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

rf_metrics = {
    "MAE": mean_absolute_error(y_test, y_pred_rf),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    "R²": r2_score(y_test, y_pred_rf),
}
print("=== Random Forest (Tuned) ===")
for k, v in rf_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
comp = pd.DataFrame({"Linear Regression": bl_metrics, "Random Forest (Tuned)": rf_metrics}).round(4)
print("=== Model Comparison ===")
comp

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, yp, name, c in [(axes[0], y_pred_bl, "Linear Regression", "#20808D"),
                         (axes[1], y_pred_rf, "Random Forest (Tuned)", "#A84B2F")]:
    ax.scatter(y_test, yp, alpha=0.3, s=15, color=c, edgecolors="none")
    lims = [0, max(y_test.max(), yp.max()) + 1]
    ax.plot(lims, lims, "k--", lw=1, label="Perfect prediction")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(f"Actual vs. Predicted — {name}", fontweight="bold")
    ax.set_xlabel("Actual Rings"); ax.set_ylabel("Predicted Rings"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, yp, name, c in [(axes[0], y_pred_bl, "Linear Regression", "#20808D"),
                         (axes[1], y_pred_rf, "Random Forest", "#A84B2F")]:
    resid = y_test - yp
    sns.histplot(resid, kde=True, bins=30, color=c, ax=ax)
    ax.axvline(0, color="black", linestyle="--", lw=1)
    ax.set_title(f"Residuals — {name}", fontweight="bold")
    ax.set_xlabel("Residual (Actual - Predicted)"); ax.set_ylabel("Count")
    ax.annotate(f"Mean: {resid.mean():.2f}\nStd: {resid.std():.2f}",
                xy=(0.72, 0.82), xycoords="axes fraction", fontsize=10,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.tight_layout(); plt.show()

In [ ]:
rf_model = best_rf.named_steps["model"]
feat_names = num_features + list(best_rf.named_steps["pre"]
    .named_transformers_["cat"].get_feature_names_out(cat_features))
fi = pd.Series(rf_model.feature_importances_, index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
fi.plot(kind="barh", color="#20808D", ax=ax)
ax.set_title("Feature Importances — Random Forest", fontweight="bold")
ax.set_xlabel("Importance"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# --- Disaggregated Performance Analysis by Sex Category ---
# Evaluate model fairness by computing metrics separately for each sex group

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sex_test = X_test["Sex"].values  # original sex labels from test set

results = []
for sex_label in ["M", "F", "I"]:
    mask = sex_test == sex_label
    n = mask.sum()
    if n == 0:
        continue
    y_true = y_test[mask]
    y_pred = y_pred_rf[mask]
    results.append({
        "Sex": sex_label,
        "N (test)": n,
        "MAE": round(mean_absolute_error(y_true, y_pred), 4),
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        "R²": round(r2_score(y_true, y_pred), 4),
    })

disagg = pd.DataFrame(results)
print("=== Disaggregated Performance by Sex Category (Random Forest) ===")
disagg